## Working with DGGRID in Vgrid DGGS

[![image](https://jupyterlite.rtfd.io/en/latest/_static/badge.svg)](https://demo.vgrid.vn/lab/index.html?path=vgrid/06_dggrid.ipynb)
[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeoshub/vgrid/blob/main/docs/notebooks/06_dggrid.ipynb)
[![image](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/opengeoshub/vgrid/HEAD?filepath=docs/notebooks/06_dggrid.ipynb)
[![image](https://studiolab.sagemaker.aws/studiolab.svg)](https://studiolab.sagemaker.aws/import/github/opengeoshub/vgrid/blob/main/docs/notebooks/06_dggrid.ipynb)
[![image](https://jupyterlite.rtfd.io/en/latest/_static/badge.svg)](https://jupyterlite.gishub.vn/lab/index.html?path=notebooks/vgrid/06_dggrid.ipynb)

Full Vgrid DGGS documentation is available at [vgrid document](https://vgrid.gishub.vn).

To work with Vgrid DGGS directly in GeoPandas and Pandas, please use [vgridpandas](https://pypi.org/project/vgridpandas/). Full Vgridpandas DGGS documentation is available at [vgridpandas document](https://vgridpandas.gishub.vn).

To work with Vgrid DGGS in QGIS, install the [Vgrid Plugin](https://plugins.qgis.org/plugins/vgridtools/).

To visualize DGGS in Maplibre GL JS, try the [vgrid-maplibre](https://www.npmjs.com/package/vgrid-maplibre) library.

For an interactive demo, visit the [Vgrid Homepage](https://vgrid.vn).

### Install vgrid
Uncomment the following line to install [vgrid](https://pypi.org/project/vgrid/).

In [ ]:
# %pip install vgrid --upgrade

### Download a portable version of DGGRID and start a DGGRID instance

In [ ]:
from dggrid4py import tool, DGGRIDv8

dggrid_exec = tool.get_portable_executable(".")

dggrid_instance = DGGRIDv8(
    executable=dggrid_exec,
    working_dir=".",
    capture_logs=True,
    silent=True,
    has_gdal=False,
    tmp_geo_out_legacy=True,
    debug=False,
)

### latlon2dggrid

In [ ]:
from vgrid.conversion.latlon2dggs import latlon2dggrid

dggs_type = "ISEA4D"  # choose one from ['CUSTOM', 'SUPERFUND', 'PLANETRISK', 'ISEA3H', 'ISEA4H', 'ISEA4T', 'ISEA4D', 
                            # 'ISEA43H', 'ISEA7H', 'IGEO7', 'FULLER3H', 'FULLER4H', 'FULLER4T', 'FULLER4D', 'FULLER43H', 'FULLER7H']
resolution = 10
address_type = "SEQNUM"
# "Q2DI",  # quad number and (i, j) coordinates on that quad
# "SEQNUM",  # DGGS index - linear address (1 to size-of-DGG), not supported for parameter input_address_type if dggs_aperture_type is SEQUENCE
# "PLANE",  # (x, y) coordinates on unfolded ISEA plane,  only supported for parameter output_address_type;
# "Q2DD",  # quad number and (x, y) coordinates on that quad
# "PROJTRI",  # PROJTRI - triangle number and (x, y) coordinates within that triangle on the ISEA plane
# "VERTEX2DD",  # vertex number, triangle number, and (x, y) coordinates on ISEA plane
# "Z3",  # hexadecimal characters index system Z3 especially usefull for ISEA3H
# "Z3_STRING",  # numerical digits representation of Z3 (as characters, not an integer)
# "Z7",  # hexadecimal characters index system Z7 especially usefull for ISEA7H, also in preset IGEO7
# "Z7_STRING",  # numerical digits representation of Z7 (as characters, not an integer)
# "ZORDER",  # index system ZORDER especially usefull for ISEA3H, ISEA4H and mixed aperture
# "ZORDER_STRING",  # numerical digits representation of ZORDER (as characters, not an integer)
dggrid_id = latlon2dggrid(
    dggrid_instance, dggs_type, 10.775276, 106.706797, resolution,output_address_type=address_type)
dggrid_id

### DGGRID to Polygon (GeoDataFrame)

In [ ]:
from vgrid.conversion.dggs2geo.dggrid2geo import dggrid2geo
dggrid_geo = dggrid2geo(dggrid_instance, dggs_type, dggrid_id, resolution)
dggrid_geo.plot()

### DGGRID to GeoJSON

In [ ]:
from vgrid.conversion.dggs2geo.dggrid2geo import dggrid2geojson

dggrid_geojson = dggrid2geojson(
    dggrid_instance, dggs_type, dggrid_id, resolution, address_type
)
dggrid_geojson

### Vector to DGGRID

In [ ]:
from vgrid.conversion.vector2dggs.vector2dggrid import vector2dggrid

file_path = (
    "https://raw.githubusercontent.com/opengeoshub/vopendata/main/shape/polygon2.geojson"
)
dggs_type = "ISEA4T"  # choose one from ['CUSTOM', 'SUPERFUND', 'PLANETRISK', 'ISEA3H', 'ISEA4H',
# 'ISEA4T', 'ISEA4D', 'ISEA43H', 'ISEA7H', 'IGEO7', 'FULLER3H', 'FULLER4H', 'FULLER4T', 'FULLER4D', 'FULLER43H', 'FULLER7H']
resolution = 15
vector_to_dggrid = vector2dggrid(
    dggrid_instance,
    dggs_type,
    file_path,
    resolution=resolution,
    compact=True,
    predicate="intersects",
    output_address_type=address_type,
    output_format="gpd",
    split_antimeridian=True,
    aggregate=True,
    include_properties = True
)
# Visualize the output
# vector_to_dggrid
vector_to_dggrid.plot(edgecolor="white")

### DGGRID Binning

In [ ]:
from vgrid.binning.dggridbin import dggridbin

dggs_type = "isea3h"  # choose one from ['CUSTOM', 'SUPERFUND', 'PLANETRISK', 'ISEA3H', 'ISEA4H',
# 'ISEA4T', 'ISEA4D', 'ISEA43H', 'ISEA7H', 'IGEO7', 'FULLER3H', 'FULLER4H', 'FULLER4T', 'FULLER4D', 'FULLER43H', 'FULLER7H']
file_path = (
    "https://raw.githubusercontent.com/opengeoshub/vopendata/main/csv/housing.csv"
)
stats = "max"
numeric_col="median_house_value"

dggrid_bin = dggridbin(
    dggrid_instance,
    dggs_type,
    file_path,
    resolution=10,
    stats=stats,
    numeric_col=numeric_col,
    # category_col="category",
    output_format="gpd",
)
# dggal_bin.head()
dggrid_bin.plot(
    column=f"{numeric_col}_{stats}",  # numeric column to base the colors on
    cmap="Spectral_r",  # color scheme (matplotlib colormap)
    legend=True,
    linewidth=0.2,  # boundary width (optional)
)

### DGGRID Resampling

In [ ]:
from vgrid.conversion.dggsresample import dggsresample
dggrid_resample = dggsresample(
    source_dggs=dggrid_bin,
    dggs_from=f"dggrid_{dggs_type}",
    dggs_to="a5",
    method = "nearest", # area_weighted, nearest
    dggs_col=f"dggrid_{dggs_type}",
    resample_col=f"{numeric_col}_{stats}",  # column on h3_bin with numbers
    fix_antimeridian=None,
    split_antimeridian=False,
)
dggrid_resample.plot(
    column=f"{numeric_col}_{stats}",  # numeric column to base the colors on
    cmap="Spectral_r",  # color scheme (matplotlib colormap)
    legend=True,
    linewidth=0.2,  # boundary width (optional)
)

### Raster to DGGRID

#### Download and open raster

In [ ]:
from vgrid.utils.io import download_file
import rasterio
from rasterio.plot import show

raster_url = (
    "https://raw.githubusercontent.com/opengeoshub/vopendata/main/raster/rgb.tif" # SRTM.tif
)
raster_file = download_file(raster_url)
src = rasterio.open(raster_file, "r")
print(src.meta)
show(src)

#### Convert raster to DGGRID

In [ ]:
from vgrid.conversion.raster2dggs.raster2dggrid import raster2dggrid

dggs_type = "ISEA3H"  # choose one from ['CUSTOM', 'SUPERFUND', 'PLANETRISK', 'ISEA3H', 'ISEA4H',
# 'ISEA4T', 'ISEA4D', 'ISEA43H', 'ISEA7H', 'IGEO7', 'FULLER3H', 'FULLER4H', 'FULLER4T', 'FULLER4D', 'FULLER43H', 'FULLER7H']
raster_to_dggrid = raster2dggrid(dggrid_instance, dggs_type, raster_file, resolution = 27, 
                                method = "binning", # binnning, nearest
                                stats = 'mean', output_format="gpd")

#### Visualize the output

In [ ]:
# %pip install folium

In [ ]:
import folium

m = folium.Map(tiles="CartoDB positron", max_zoom=28)

dggrid_layer = folium.GeoJson(
    raster_to_dggrid,
    style_function=lambda x: {
        "fillColor": f"rgb({x['properties']['band_1']}, {x['properties']['band_2']}, {x['properties']['band_3']})",
        "fillOpacity": 1,
        "color": "black",
        "weight": 0.5,
    },
    popup=folium.GeoJsonPopup(
        fields=[ f"dggrid_{dggs_type}", "resolution", "band_1", "band_2", "band_3", "cell_area"],
        aliases=["DGGRID ID", "Resolution", "Band 1", "Band 2", "Band 3", "Area (m²)"],
        style="""
            background-color: white;
            border: 2px solid black;
            border-radius: 3px;
            box-shadow: 3px;
        """,
    ),
).add_to(m)

m.fit_bounds(dggrid_layer.get_bounds())

# Display the map
m

### DGGRID Generator

In [ ]:
from vgrid.generator.dggridgen import dggridgen

dggs_type = "IGEO7"  # choose one from ['CUSTOM', 'SUPERFUND', 'PLANETRISK', 'ISEA3H', 'ISEA4H', 'ISEA4T',
# 'ISEA4D', 'ISEA43H', 'ISEA7H', 'IGEO7', 'FULLER3H', 'FULLER4H', 'FULLER4T', 'FULLER4D', 'FULLER43H', 'FULLER7H']
resolution = 1
# bbox = [106.699007, 10.762811, 106.717674, 10.778649]  # (min_lon, min_lat, max_lon, max_lat)
options = {"densification": 30}
dggrid_gen = dggridgen(
                dggrid_instance, dggs_type=dggs_type, resolution=resolution, 
                split_antimeridian = True, aggregate=True, options=options
                )
dggrid_gen.plot(edgecolor="white")

### DGGRID Inspect

In [ ]:
from vgrid.stats.dggridstats import dggridinspect

dggs_type = "IGEO7"  # choose one from ['CUSTOM', 'SUPERFUND', 'PLANETRISK', 'ISEA3H', 'ISEA4H', 'ISEA4T',
# 'ISEA4D', 'ISEA43H', 'ISEA7H', 'IGEO7', 'FULLER3H', 'FULLER4H', 'FULLER4T', 'FULLER4D', 'FULLER43H', 'FULLER7H']
resolution = 2
options = {"densification": 30}
dggrid_inspect = dggridinspect(dggrid_instance, dggs_type, resolution, options= options)

dggrid_inspect.head() 

### DGGRID Normalized Area Histogram

In [ ]:
from vgrid.stats.dggridstats import dggrid_norm_area_hist

dggrid_norm_area_hist(dggs_type, dggrid_inspect)

### Distribution of DGGRID Area Distortions

In [ ]:
from vgrid.stats.dggridstats import dggrid_norm_area

dggrid_norm_area(dggs_type, dggrid_inspect, crs="proj=moll")

### DGGRID IPQ Compactness Histogram

Isoperimetric Inequality (IPQ) Compactness (suggested by [Osserman, 1978](https://sites.math.washington.edu/~toro/Courses/20-21/MSF/osserman.pdf)):

$$C_{IPQ} = \frac{4 \pi A}{p^2}$$
The range of the IPQ compactness metric is [0,1].

A circle represents the maximum compactness with a value of 1.

As shapes become more irregular or elongated, their compactness decreases toward 0.

In [ ]:
from vgrid.stats.dggridstats import dggrid_compactness_ipq_hist

dggrid_compactness_ipq_hist(dggs_type, dggrid_inspect)

### Distribution of DGGRID IPQ Compactness

In [ ]:
from vgrid.stats.dggridstats import dggrid_compactness_ipq

dggrid_compactness_ipq(dggs_type, dggrid_inspect, crs="proj=moll")

### DGGRID Convex hull Compactness Histogram:

$$C_{CVH} = \frac{A}{A_{CVH}}$$


The range of the convex hull compactness metric is [0,1].

As shapes become more concave, their convex hull compactness decreases toward 0.

In [ ]:
from vgrid.stats.dggridstats import dggrid_compactness_cvh_hist

dggrid_compactness_cvh_hist(dggs_type, dggrid_inspect)

### Distribution of DGGRID Convex hull Compactness

In [ ]:
from vgrid.stats.dggridstats import dggrid_compactness_cvh

dggrid_compactness_cvh(dggs_type, dggrid_inspect)

### DGGRID Statistics

Characteristic Length Scale (CLS - suggested by Ralph Kahn): the diameter of a spherical cap of the same cell's area

In [ ]:
from vgrid.stats.dggridstats import dggridstats

dggs_type = "IGEO7"  # choose one from ['CUSTOM', 'SUPERFUND', 'PLANETRISK', 'ISEA3H', 'ISEA4H', 'ISEA4T',
# 'ISEA4D', 'ISEA43H', 'ISEA7H', 'IGEO7', 'FULLER3H', 'FULLER4H', 'FULLER4T', 'FULLER4D', 'FULLER43H', 'FULLER7H']
dggrid_stats = dggridstats(dggrid_instance, dggs_type, "m")
dggrid_stats